# Parte 3

**Tarefas 11 a 16**

O objetivo desta parte do trabalho é estudar o comportamento dos otimizadores de consulta dos SGBDs através do exame e análise dos planos de execução para consultas SQL sobre tabelas que serão fornecidos. Será bastante utilizado o comando `EXPLAIN ANALYZE`, que permite visualizar todas as etapas envolvidas no processamento de uma consulta. Usaremos para isso a tabela [movies](https://drive.google.com/file/d/1W6wovSsVu4B0OIo_tsSBBHi8WRKQqnat/view)

Configuração Inicial:

In [19]:
# Conectar ao banco de dados PostgreSQL
import psycopg2

config = {
    'dbname': 'icomp',
    'user': 'icomp',
    'password': 'icomp123',
}

conn = psycopg2.connect(**config)
cur = conn.cursor()

# Configurar rich
from rich.table import Table
from rich.console import Console
from rich.syntax import Syntax
from rich.panel import Panel

console = Console()

---
## Tarefa 11
**Preparação e Verificação do Ambiente**

a) Execute o script movie.sql em movies para criar as tabelas e índices e carregar os dados necessários às próximas atividades

b) Verifique no catálogo do banco de dados os seguintes metadados sobre os índices associados às tabelas e apresente-os no relatório: Nome do índice, nome da tabela, altura, número máximo de chaves por bloco, número médio de chaves por bloco, número de blocos folha, número de médio de blocos folha por chave, número médio de blocos de dados por chave, número de linhas e número de chaves distintas.



### O que entregar
**Relatório com os resultados da verificação**

#### **A)** Execute o script `movie.sql` em movies para criar as tabelas e índices e carregar os dados necessários às próximas atividades

In [2]:
with open('movie.sql', 'r') as f:
    sql_script = f.read()
    cur.execute(sql_script)

# Checar se a tabela foi criada e os dados foram inseridos
cur.execute("""SELECT COUNT(*) FROM movie;""")
cur.fetchall()

[(1844,)]

#### **B)** Verifique no catálogo do banco de dados os seguintes metadados sobre os índices associados às tabelas e apresente-os no relatório: Nome do índice, nome da tabela, altura, número máximo de chaves por bloco, número médio de chaves por bloco, número de blocos folha, número de médio de blocos folha por chave, número médio de blocos de dados por chave, número de linhas e número de chaves distintas. 

Os metadados que são precisos para analisar índices da tabela `movie` são obtidos a partir dos seguintes catálogos:
- `pg_class`: fornece informações sobre tabelas e índices e armazena estatísticas básicas
- `pg_index`: relaciona cada índice à tabela correspondente
- `pg_attribute`: lista as colunas de tabelas e índices
- `pg_namespace`: define os esquemas do banco
- `pg_stat_all_indexes`: dá estatístiacs agregadas de uso dos índices

Algumas métricas específicas de organização física (altura da árvore, blocos folha, densidade de chaves etc.) só podem ser extraídas com a função `pgstatindex()`, fornecida pela extensão `pgstattuple`. É preciso que essa extensão esteja instalada e habilitada no BD. 

In [3]:
cur.execute("CREATE EXTENSION IF NOT EXISTS pgstattuple;")
conn.commit()

Identificamos todos os índices associados à tabela.  
Isso vem exclusivamente dos catálogos:

- `pg_class` para nomes de índices e tabelas  
- `pg_index` para relacionamentos  
- `pg_namespace` para filtrar o schema público

In [4]:
cur.execute("""
SELECT
    idx.relname AS index_name,
    tbl.relname AS table_name,
    pg_get_indexdef(i.indexrelid) AS index_def
FROM pg_index i
JOIN pg_class idx ON idx.oid = i.indexrelid
JOIN pg_class tbl ON tbl.oid = i.indrelid
JOIN pg_namespace n ON n.oid = tbl.relnamespace
WHERE n.nspname = 'public'
  AND tbl.relname = 'movie';
""")

indexes = cur.fetchall()

table = Table(title="Índices da tabela `movie`", title_style="bold bright_white")
table.add_column("Índice", header_style="bold bright_cyan")
table.add_column("Tabela", header_style="bold bright_cyan")
table.add_column("Definição", header_style="bold bright_cyan", overflow="fold")

for index_name, table_name, index_def in indexes:
    syntax = Syntax(index_def, "sql", theme="dracula", word_wrap=True)
    table.add_row(index_name, table_name, syntax)

console.print(table)

                                Índices da tabela `movie`                                
┏━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Índice      ┃ Tabela ┃ Definição                                                      ┃
┡━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ movie_key   │ movie  │ CREATE UNIQUE INDEX movie_key ON public.movie USING btree (id) │
│ movie_title │ movie  │ CREATE INDEX movie_title ON public.movie USING btree (title)   │
│ movie_votes │ movie  │ CREATE INDEX movie_votes ON public.movie USING btree (votes)   │
└─────────────┴────────┴────────────────────────────────────────────────────────────────┘

**Estatísticas internas dos índices (`pgstatindex`)**

A extensão `pgstattuple` fornece a função `pgstatindex()`, que nos revela
as propriedades internas da árvore B-Tree:

- `tree_level`: altura do índice  
- `leaf_pages`: quantidade de páginas folha  
- `avg_leaf_density`: densidade média de chaves nas folhas  
- `internal_pages`: páginas internas  
- `index_size`: tamanho total em bytes  
- `leaf_fragmentation`: fragmentação interna  

In [5]:
stats_per_index = {}

for idx, tbl, _ in indexes:
    cur.execute("SELECT * FROM pgstatindex(%s::regclass);", (idx,))
    row = cur.fetchone()
    cols = [d[0] for d in cur.description]
    stats_per_index[idx] = dict(zip(cols, row))

# mostrar estatísticas individualmente
for idx, stats in stats_per_index.items():
    t = Table(
        title=f"Estatísticas internas do índice: {idx}",
        title_style="bold bright_white"
    )
    t.add_column("Campo", header_style="bold bright_cyan")
    t.add_column("Valor", header_style="bold bright_cyan")

    for k, v in stats.items():
        t.add_row(k, str(v))

    console.print(t)

   Estatísticas internas do   
      índice: movie_key       
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 57344 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 5     │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 90.73 │
│ leaf_fragmentation │ 0.0   │
└────────────────────┴───────┘

   Estatísticas internas do   
     índice: movie_title      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 98304 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 10    │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 72.33 │
│ leaf_fragmentation │ 40.0  │
└────────────────────┴───────┘

   Estatísticas internas do   
     índice: movie_votes      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 90112 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 9     │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 47.81 │
│ leaf_fragmentation │ 77.78 │
└────────────────────┴───────┘

A tabela abaixo reúne todas as informações solicitadas no enunciado:

- Nome do índice  
- Nome da tabela  
- Altura (tree_level)  
- Número máximo de chaves por bloco (estimado para PG ≥ 12)  
- Número médio de chaves por bloco (avg_leaf_density)  
- Número de blocos folha  
- Número médio de blocos folha por chave  
- Número médio de blocos de dados por chave (estimado)  
- Número de linhas  
- Número de chaves distintas  

As estimativas para `max_keys_per_block` e `avg_data_pages_per_key` são necessárias
porque o PostgreSQL moderno não expõe mais esses campos diretamente, mas podem
ser derivadas com base nos conceitos originais da função `pgstatindex()` e no tamanho
real da tabela.

In [6]:
cur.execute("SHOW block_size;")
BLOCK_SIZE = int(cur.fetchone()[0])
print(f"Tamanho do bloco: {BLOCK_SIZE} bytes")

Tamanho do bloco: 8192 bytes


In [7]:
rows = []

for idx_name, tbl_name, idx_def in indexes:

    stats = stats_per_index[idx_name]

    # total de linhas da tabela
    cur.execute(f"SELECT COUNT(*) FROM {tbl_name};")
    total_rows = cur.fetchone()[0]

    # colunas do índice
    cur.execute("""
        SELECT a.attname
        FROM pg_attribute a
        JOIN pg_index i ON a.attrelid = i.indrelid
                       AND a.attnum = ANY(i.indkey)
        JOIN pg_class c ON c.oid = i.indexrelid
        WHERE c.relname = %s;
    """, (idx_name,))
    cols = [r[0] for r in cur.fetchall()]
    col_expr = ", ".join(cols)

    # número de chaves distintas
    cur.execute(f"SELECT COUNT(DISTINCT ({col_expr})) FROM {tbl_name};")
    distinct = cur.fetchone()[0]

    # páginas folha
    leaf_pages = stats["leaf_pages"]

    # média de blocos folha por chave (exato)
    avg_leaf_pages_per_key = (
        leaf_pages / distinct if distinct > 0 else None
    )

    # estimativa para max_leaf_keys
    # capacidade máxima ≈ densidade * espaço útil
    max_leaf_keys_estimated = int((stats["avg_leaf_density"] / 100) * 8192)

    # estimativa para média de páginas de dados por chave
    cur.execute(f"SELECT pg_relation_size('{tbl_name}'::regclass);")
    table_size_bytes = cur.fetchone()[0]
    heap_pages = table_size_bytes / BLOCK_SIZE

    avg_data_pages_per_key_estimated = (
        heap_pages / distinct if distinct > 0 else None
    )

    rows.append({
        "index_name": idx_name,
        "table_name": tbl_name,
        "height": stats["tree_level"],
        "max_keys_per_block": max_leaf_keys_estimated,
        "avg_keys_per_block": stats["avg_leaf_density"],
        "leaf_pages": leaf_pages,
        "avg_leaf_pages_per_key": avg_leaf_pages_per_key,
        "avg_data_pages_per_key": avg_data_pages_per_key_estimated,
        "total_rows": total_rows,
        "distinct_keys": distinct
    })

In [8]:
table = Table(
    title="Metadados Consolidados dos Índices da Tabela `movie`",
    title_style="bold bright_white"
)

# colunas com estilo claro e consistência
columns = [
    "Índice",
    "Tabela",
    "Altura",
    "Máx. chaves/bloco (est.)",
    "Méd. chaves/bloco",
    "Blocos folha",
    "Méd. bloco folha/chave",
    "Méd. bloco dados/chave (est.)",
    "Linhas",
    "Chaves distintas"
]

for col in columns:
    table.add_column(col, header_style="bold bright_cyan", overflow="fold")

# preenchimento da tabela
for r in rows:
    table.add_row(
        r["index_name"],
        r["table_name"],
        str(r["height"]),
        f"{r['max_keys_per_block']:.2f}",
        f"{r['avg_keys_per_block']:.2f}",
        str(r["leaf_pages"]),
        f"{r['avg_leaf_pages_per_key']:.4f}",
        f"{r['avg_data_pages_per_key']:.4f}",
        str(r["total_rows"]),
        str(r["distinct_keys"])
    )

console.print(table)

                               Metadados Consolidados dos Índices da Tabela `movie`                                
┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃           ┃        ┃        ┃ Máx.      ┃            ┃           ┃            ┃ Méd.      ┃        ┃            ┃
┃           ┃        ┃        ┃ chaves/bl ┃ Méd.       ┃           ┃ Méd. bloco ┃ bloco     ┃        ┃            ┃
┃           ┃        ┃        ┃ oco       ┃ chaves/blo ┃ Blocos    ┃ folha/chav ┃ dados/cha ┃        ┃ Chaves     ┃
┃ Índice    ┃ Tabela ┃ Altura ┃ (est.)    ┃ co         ┃ folha     ┃ e          ┃ ve (est.) ┃ Linhas ┃ distintas  ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ movie_key │ movie  │ 1      │ 7432.00   │ 90.73      │ 5         │ 0.0027     │ 0.0081    │ 1844   │ 1844       │
│ movie_tit │ movie  │ 1      │ 5925.00   │ 72.33      │ 10        │ 0.0055     │ 0.0082    │ 1844   │ 1833       │
│ le        │        │        │           │            │           │            │           │        │            │
│ movie_vot │ movie  │ 1      │ 3916.00   │ 47.81      │ 9         │ 0.0061     │ 0.0101    │ 1844   │ 1481       │
│ es        │        │        │           │            │           │            │           │        │            │
└───────────┴────────┴────────┴───────────┴────────────┴───────────┴────────────┴───────────┴────────┴────────────┘

---
## Tarefa 12
**Consultas por intervalo e índices secundários**

a) Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número pequeno de tuplas (<10 tuplas); Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

b) Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número grande de tuplas (>80% das tuplas). Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

c) Explique porque o índice sobre VOTES não é sempre usado nas consultas sobre este atributo


### O que entregar
**Relatório com as respostas das questões**


#### **A)** Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número pequeno de tuplas (<10 tuplas); Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

Pegar 10 tuplas distintas a partir do 50º registro ordenado por votos para ter uma noção de poucas tuplas.

In [18]:
cur.execute("""SELECT DISTINCT votes FROM movie ORDER BY votes LIMIT 10 OFFSET 50;""")
cur.fetchall()

[(802,),
 (803,),
 (805,),
 (806,),
 (807,),
 (808,),
 (809,),
 (811,),
 (812,),
 (813,)]

Assim, escolhemos um itervalo restrito entre 800 e 810 votos:
```postgresql
SELECT * FROM movie WHERE votes BETWEEN 800 AND 810;
```

In [21]:
query_small = """
EXPLAIN ANALYZE
SELECT *
FROM movie
WHERE votes BETWEEN 900000 AND 901000;
"""

cur.execute(query_small)
plan_small = "\n".join(row[0] for row in cur.fetchall())

console.print(Panel(plan_small, title="Plano — Consulta Seletiva (<10 tuplas)"))

╭──────────────────────────────────── Plano — Consulta Seletiva (<10 tuplas) ─────────────────────────────────────╮
│ Index Scan using movie_votes on movie  (cost=0.28..8.30 rows=1 width=30) (actual time=0.004..0.005 rows=0       │
│ loops=1)                                                                                                        │
│   Index Cond: ((votes >= 900000) AND (votes <= 901000))                                                         │
│ Planning Time: 0.165 ms                                                                                         │
│ Execution Time: 0.025 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dessa forma, conseguimos ver que o plano de execução utiliza o índice sobre `votes` para buscar as tuplas desejadas.

#### **B)** Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número grande de tuplas (>80% das tuplas). Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

In [22]:
query_big = """
EXPLAIN ANALYZE
SELECT *
FROM movie
WHERE votes > 100;"""

cur.execute(query_big)
plan_big = "\n".join(row[0] for row in cur.fetchall())

console.print(Panel(plan_big, title="Plano — Consulta Não Seletiva (>80% tuplas)"))

╭────────────────────────────────── Plano — Consulta Não Seletiva (>80% tuplas) ──────────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..38.05 rows=1844 width=30) (actual time=0.013..0.363 rows=1844 loops=1)           │
│   Filter: (votes > 100)                                                                                         │
│ Planning Time: 0.249 ms                                                                                         │
│ Execution Time: 0.438 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Em consultas não seletivas, o otimizador opta por não usar o índice, realizando um **seq scan** na tabela.

#### **C)** Explique porque o índice sobre VOTES não é sempre usado nas consultas sobre este atributo

O índice secundário sobre `votes` não é sempre utilizado devido às regras de custo do otimizador do SGBD.

Para retornar poucas tuplas (< 5–10%), é mais barato caminhar pela B-tree outraçar um intervalo nas folhas da árvore e buscar as tuplas correspondentes no heap.

Quando a consulta retorna muitas tuplas (> 30–40%), o custo muda. O SGBD teria que:
- seguir muitas folhas da B-tree  
- fazer milhares de acessos aleatórios ao heap  

Essa forma acaba sendo mais cara que uma varredura, que lê as páginas da tabela sequencialmente.

---
## Tarefa 13
**Comparações de operadores de agregação**

Considere as seguintes consultas em SQL, sobre o atributo VOTES, as quais são equivalentes:

```postgresql
SELECT title FROM movie WHERE votes >= (SELECT MAX(votes) FROM movie);
```

```postgresql 
SELECT title FROM movie WHERE votes >= ALL (SELECT votes FROM movie);
```

a) Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as duas consultas acima

b) Existe alguma diferença entre os planos de consultas? Qual das duas é mais eficiente? Explique


### O que entregar
**Relatório com as respostas das questões**

#### **A)** Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as duas consultas acima

In [23]:
query1 = """
EXPLAIN ANALYZE
SELECT title
FROM movie
WHERE votes >= (SELECT MAX(votes) FROM movie);
"""

cur.execute(query1)
plan1 = "\n".join(row[0] for row in cur.fetchall())

console.print(
    Panel(plan1, 
    title="Plano — Consulta 1 (votes >= (SELECT MAX(votes)))")
)

╭─────────────────────────────── Plano — Consulta 1 (votes >= (SELECT MAX(votes))) ───────────────────────────────╮
│ Index Scan using movie_votes on movie  (cost=0.61..35.37 rows=615 width=16) (actual time=0.153..0.156 rows=1    │
│ loops=1)                                                                                                        │
│   Index Cond: (votes >= $1)                                                                                     │
│   InitPlan 2 (returns $1)                                                                                       │
│     ->  Result  (cost=0.32..0.33 rows=1 width=4) (actual time=0.075..0.076 rows=1 loops=1)                      │
│           InitPlan 1 (returns $0)                                                                               │
│             ->  Limit  (cost=0.28..0.32 rows=1 width=4) (actual time=0.071..0.071 rows=1 loops=1)               │
│                   ->  Index Only Scan Backward using movie_votes on movie movie_1  (cost=0.28..76.55 rows=1844  │
│ width=4) (actual time=0.069..0.070 rows=1 loops=1)                                                              │
│                         Index Cond: (votes IS NOT NULL)                                                         │
│                         Heap Fetches: 0                                                                         │
│ Planning Time: 1.135 ms                                                                                         │
│ Execution Time: 0.912 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
query2 = """
EXPLAIN ANALYZE
SELECT title
FROM movie
WHERE votes >= ALL (SELECT votes FROM movie);
"""

cur.execute(query2)
plan2 = "\n".join(row[0] for row in cur.fetchall())

console.print(
    Panel(plan2, 
    title="Plano — Consulta 2 (votes >= ALL (SELECT votes))")
)

╭─────────────────────────────── Plano — Consulta 2 (votes >= ALL (SELECT votes)) ────────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..43620.99 rows=922 width=16) (actual time=0.895..1.355 rows=1 loops=1)            │
│   Filter: (SubPlan 1)                                                                                           │
│   Rows Removed by Filter: 1843                                                                                  │
│   SubPlan 1                                                                                                     │
│     ->  Materialize  (cost=0.00..42.66 rows=1844 width=4) (actual time=0.000..0.000 rows=2 loops=1844)          │
│           ->  Seq Scan on movie movie_1  (cost=0.00..33.44 rows=1844 width=4) (actual time=0.002..0.392         │
│ rows=1844 loops=1)                                                                                              │
│ Planning Time: 0.126 ms                                                                                         │
│ Execution Time: 1.412 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### **B)** Existe alguma diferença entre os planos de consultas? Qual das duas é mais eficiente? Explique

Consulta 1:
- Calcula `MAX(votes)` usando um 'Index Only Scan Backward`
- Usa um `Index Scan` em `movie_votes` para encontrar apenas as tuplas desejadas
- Tempo de execução esperada: **~0.9 ms**

Consulta 2:
- Em vez de reescrever a expressão `>= ALL` como `MAX`, executou uma varredura completa na tabela
- Para cada linha, verificou a condição usando um `Sub Plan`
- Esse subplano é executado p/ cada linha (1844), resultando em um custo muito maior
- Tempo de execução esperada: **~1.4 ms**

O plano da segunda consulta é pior, visto que ela faz uma leitura sequencial completa da tabela, ignorando o índice. A primeira consulta é mais eficiente, pois calcula o valor máximo uma vez e usa o índice para buscar as tuplas correspondentes.

---
## Tarefa 14
**Consultas com Junção e Seleção**

Considere as duas consultas equivalentes em SQL a seguir, as quais retornam os filmes com mais votos que “Star Wars”:

```postgresql 
SELECT title FROM movie WHERE votes > (SELECT votes FROM movie WHERE title = 'Star Wars');
```

```postgresql 
SELECT m1.title FROM movie m1, movie m2 WHERE m1.votes > m2.votes AND m2.title = 'Star Wars';
```
 
a) Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as duas consultas acima

b) Existe alguma diferença entre os planos de consultas? Qual das duas é mais eficiente? Explique


### O que entregar
**Relatório com as respostas das questões**

#### **A)** Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as duas consultas acima

In [25]:
queries = [
    """
EXPLAIN ANALYZE
SELECT title
FROM movie
WHERE votes > (SELECT votes FROM movie WHERE title = 'Star Wars');
    """,
    """
EXPLAIN ANALYZE
SELECT m1.title
FROM movie m1, movie m2
WHERE m1.votes > m2.votes AND m2.title = 'Star Wars';
    """
]

for i, query in enumerate(queries):
    cur.execute(query)
    plan = "\n".join(row[0] for row in cur.fetchall())
    console.print(Panel(plan, title=f"Plano da Consulta {i+1} — Consulta com Junção e Seleção"))

╭────────────────────────────── Plano da Consulta 1 — Consulta com Junção e Seleção ──────────────────────────────╮
│ Index Scan using movie_votes on movie  (cost=8.57..43.34 rows=615 width=16) (actual time=0.199..0.200 rows=0    │
│ loops=1)                                                                                                        │
│   Index Cond: (votes > $0)                                                                                      │
│   InitPlan 1 (returns $0)                                                                                       │
│     ->  Index Scan using movie_title on movie movie_1  (cost=0.28..8.29 rows=1 width=4) (actual                 │
│ time=0.119..0.120 rows=1 loops=1)                                                                               │
│           Index Cond: ((title)::text = 'Star Wars'::text)                                                       │
│ Planning Time: 1.339 ms                                                                                         │
│ Execution Time: 1.019 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── Plano da Consulta 2 — Consulta com Junção e Seleção ──────────────────────────────╮
│ Nested Loop  (cost=0.56..49.49 rows=615 width=16) (actual time=0.040..0.041 rows=0 loops=1)                     │
│   ->  Index Scan using movie_title on movie m2  (cost=0.28..8.29 rows=1 width=4) (actual time=0.031..0.032      │
│ rows=1 loops=1)                                                                                                 │
│         Index Cond: ((title)::text = 'Star Wars'::text)                                                         │
│   ->  Index Scan using movie_votes on movie m1  (cost=0.28..35.04 rows=615 width=20) (actual time=0.004..0.004  │
│ rows=0 loops=1)                                                                                                 │
│         Index Cond: (votes > m2.votes)                                                                          │
│ Planning Time: 1.064 ms                                                                                         │
│ Execution Time: 0.133 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### **B)** Existe alguma diferença entre os planos de consultas? Qual das duas é mais eficiente? Explique


Há diferenças claras entre os planos de execução

**Consulta 1:**
- Primeiro executa a subquery `SELECT votes FROM movie WHERE title = 'Star Wars'`
    - Essa subconsulta retorna **apenas um único valor**
    - O SGBD geralmente resolve isso com um scan no índice `movie_title` ou um `Seq Scan` curto
- Com esse valor em mãos, a consulta principal vira:
    - `votes > valor_unico`
- O otimizador pode usar o índice `movie_votes` diretamente
- O plano é simples  

Há apenas um acesso para buscar o valor de *Star Wars* e depois um único `Index Scan` na tabela

**Consulta 2:**
- A tabela `movie` aparece duas vezes com uma combinação
- Faz o filtro do join para pegar `m1.votes > m2.votes` e `m2.title = 'Star Wars'`
- O plano precisa ser estruturado como uma junção
    - Pode envolver `Nested Loop`
    - O otimizador precisa considerar cardinalidades, filtros e custo de join
- Geralmente gera **mais operadores** que a consulta 1 (self-join lida com `n^2` comparações, n sendo o número de linhas)


A consulta 1 (subconsulta escalar) é mais eficiente, visto que ela permite que o otimizador obtenha o valro de votos de *Star Wars* uma única vez e transforme a consulta em uma simples comparação, enquanto a segunda consulta força um self-join, que é mais custoso

---
## Tarefa 15
**Casamento de Strings e Índices**

Considere as seguintes consultas SQL sobre o atributo  TITLE usando o operador `LIKE`:

```postgresql 
SELECT title FROM movie WHERE title LIKE 'I%';
```

```postgresql 
SELECT title FROM movie WHERE substr(title, 1, 1) = 'I';
```

```postgresql 
SELECT title FROM movie WHERE title LIKE '%A'; 
```

a) Apresente o resultado do comando explain sobre as três consultas acima

b) Qual das três apresenta o menor custo? Porque?
 
c) O índice sobre TITLE foi usado para todas elas? Justifique.


### O que entregar
**Relatório com as respostas das questões**

#### **A)** Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as três consultas acima

In [30]:
queries = [
    """
EXPLAIN ANALYZE
SELECT title FROM movie WHERE substr(title, 1, 1) = 'I';
    """,
    """
EXPLAIN ANALYZE
SELECT title FROM movie WHERE title LIKE '%A';
    """,
    """
EXPLAIN ANALYZE
SELECT title FROM movie WHERE title LIKE 'I%';
    """,
]

for i, query in enumerate(queries):
    cur.execute(query)
    plan = "\n".join(row[0] for row in cur.fetchall())
    console.print(Panel(plan, title=f"Plano da Consulta {i+1} — Casamento de Strings e Índices"))

╭───────────────────────────── Plano da Consulta 1 — Casamento de Strings e Índices ──────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..42.66 rows=9 width=16) (actual time=0.052..0.500 rows=25 loops=1)                │
│   Filter: (substr((title)::text, 1, 1) = 'I'::text)                                                             │
│   Rows Removed by Filter: 1819                                                                                  │
│ Planning Time: 0.193 ms                                                                                         │
│ Execution Time: 0.617 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── Plano da Consulta 2 — Casamento de Strings e Índices ──────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..38.05 rows=37 width=16) (actual time=0.010..0.220 rows=30 loops=1)               │
│   Filter: ((title)::text ~~ '%A'::text)                                                                         │
│   Rows Removed by Filter: 1814                                                                                  │
│ Planning Time: 0.095 ms                                                                                         │
│ Execution Time: 0.228 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── Plano da Consulta 3 — Casamento de Strings e Índices ──────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..38.05 rows=18 width=16) (actual time=0.006..0.164 rows=25 loops=1)               │
│   Filter: ((title)::text ~~ 'I%'::text)                                                                         │
│   Rows Removed by Filter: 1819                                                                                  │
│ Planning Time: 0.027 ms                                                                                         │
│ Execution Time: 0.168 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### **B)** Qual das três apresenta o menor custo? Porque?

| Consulta                               | Custo Estimado |
|----------------------------------------|----------------|
| Consulta 2 (substr(title, 1, 1) = 'I') | 0.00 .. 42.66  |
| Consulta 1 (LIKE 'I%')                 | 0.00 .. 38.05  |
| Consulta 3 (LIKE '%A')                 | 0.00 .. 38.05  |

O planejador acredita que a consulta 1 será a mais cara, enquanto a 2 e a 3 terão o mesmo custo.

Isso se dá porque o planejador estima o custo com base na função de filtro aplicada a cada linha. Ele considera que:
- `substr()` é mais caro que `LIKE`, visto que executa uma função adicional de manipulação de strings
- `LIKE '%A'` e `LIKE 'I%'` são equivalentes em custo

No caso do tempo real (Actual Time):
| Consulta                               | Tempo Real (ms) |
|----------------------------------------|-----------------|
| Consulta 1 (substr(title, 1, 1) = 'I') | 0.052 .. 0.5    |
| Consulta 2 (LIKE 'I%')                 | 0.01 .. 0.22    |
| Consulta 3 (LIKE '%A')                 | 0.006 .. 0.164  |

O custo maior da consulta 1 (42.66) ocorre porque substr() é uma função aplicada em cada linha, que estima um maior uso de CPU por linha.
Os custos iguais de 2 e 3 (38.05) ocorrem porque o operador LIKE, para o planejador, tem o mesmo custo fixo (o padrão específico não influencia a estimativa)


#### **C)** O índice sobre TITLE foi usado para todas elas? Justifique.

Não, todas as consutlas utilizaram `seq scan` na tabela `movie`.

Primeiramente, no caso da `substr`, o uso de índice não faz sentido, visto que a função precisa ser aplicada em cada linha, o que impede o uso do índice. O SGBD não tem como transformar a expressão de substring em algo indexável.

Para os casos de `LIKE`, o índice poderia ser usado para `LIKE 'I%'`, visto que o padrão começa com um prefixo fixo. No entanto, o planejador optou por não usar o índice, provavelmente porque a seletividade esperada (número de linhas retornadas) era alta o suficiente para que um `seq scan` fosse mais eficiente do que o custo de acessar o índice.

Para o caso de `LIKE '%A'`, seria necessário realizar uma varredura completa, visto que o padrão começa com qualquer coisa, o que impede o uso do índice (não há caminhamento de nós na B-tree que irá ajudar a encontrar mais rápido). Portanto, o `seq scan` é a única opção viável nesse caso.

---
## Tarefa 16
**Verificação da hipótese de distribuição uniforme na estimativa de seletividade**


Considere as seguintes  consultas sobre o atributo TITLE da tabela MOVIE:

```postgresql
SELECT title FROM movie WHERE votes < 1000;
```

```postgresql 
SELECT title FROM movie WHERE votes > 40000;
``` 

a) Apresente o resultado do comando explain sobre as duas consultas acima. Explique o resultado.

b) Compare o número de tuplas selecionadas por cada consulta. Qual das duas tem a menor seletividade?



### O que entregar
**Relatório com as respostas das questões**

#### **A)** Apresente o resultado do comando `EXPLAIN ANALYZE` sobre as duas consultas acima. Explique o resultado.

In [31]:
queries = [
    """
EXPLAIN ANALYZE
SELECT title FROM movie WHERE votes < 1000;
    """,
    """
EXPLAIN ANALYZE
SELECT title FROM movie WHERE votes > 40000;
    """,
]

for i, query in enumerate(queries):
    cur.execute(query)
    plan = "\n".join(row[0] for row in cur.fetchall())
    console.print(Panel(plan, title=f"Plano da Consulta {i+1} — Verificação da Hipótese de Distribuição Uniforme"))

╭──────────────────── Plano da Consulta 1 — Verificação da Hipótese de Distribuição Uniforme ─────────────────────╮
│ Index Scan using movie_votes on movie  (cost=0.28..20.02 rows=328 width=16) (actual time=0.073..0.194 rows=326  │
│ loops=1)                                                                                                        │
│   Index Cond: (votes < 1000)                                                                                    │
│ Planning Time: 0.149 ms                                                                                         │
│ Execution Time: 0.495 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────── Plano da Consulta 2 — Verificação da Hipótese de Distribuição Uniforme ─────────────────────╮
│ Index Scan using movie_votes on movie  (cost=0.28..8.42 rows=8 width=16) (actual time=0.011..0.012 rows=4       │
│ loops=1)                                                                                                        │
│   Index Cond: (votes > 40000)                                                                                   │
│ Planning Time: 0.075 ms                                                                                         │
│ Execution Time: 0.021 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Para a primeira consulta (`votes < 1000`), o plano de execução mostra que o otimizador escolheu um `Index Scan` no índice `movie_votes`. Isso indica que a consulta é seletiva o suficiente para justificar o uso do índice, retornando um número relativamente pequeno de tuplas.

Da mesma forma, a SGBD avalia que a segunda consulta (`votes > 40000`) também é seletiva, optando por um `Index Scan` no mesmo índice. Isso sugere que ambas as consultas retornam um número limitado de tuplas, permitindo o uso eficiente do índice.

#### **B)** Compare o número de tuplas selecionadas por cada consulta. Qual das duas tem a menor seletividade?

In [33]:
# executando as consultas para contar o número de tuplas retornadas
cur.execute("SELECT COUNT(*) FROM movie WHERE votes < 1000;")
count_less_1000 = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM movie WHERE votes > 40000;")
count_greater_40000 = cur.fetchone()[0]

table = Table(
    title="Número de Tuplas Retornadas por Consulta",
    title_style="bold bright_white"
)
table.add_column("Consulta", header_style="bold bright_cyan")
table.add_column("Número de Tuplas", header_style="bold bright_cyan")
table.add_row("votes < 1000", str(count_less_1000))
table.add_row("votes > 40000", str(count_greater_40000))
console.print(table)

  Número de Tuplas Retornadas por   
              Consulta              
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Consulta      ┃ Número de Tuplas ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ votes < 1000  │ 326              │
│ votes > 40000 │ 4                │
└───────────────┴──────────────────┘

A consulta de `votes < 1000` retornou **326 tuplas**, enquanto a consulta de `votes > 40000` retornou apenas **4**.

Dessa forma, a consulta `votes > 40000` é a mais seletiva, visto que retorna um número muito menor de tuplas em comparação com a outra consulta.